# 01 — GLORYS12 monthly subset over the `egsl` envelope

Downloads monthly-mean surface ocean and sea-ice fields from the Copernicus Marine reanalysis
`GLOBAL_MULTIYEAR_PHY_001_030` (GLORYS12V1) over the lon/lat bounds of ice-clim's `egsl` mask.

In [ ]:
from pathlib import Path

import copernicusmarine
import geopandas as gpd

EGSL_MASK_PATH = Path.home() / "data/masks/climatology_bbox/egsl/egsl_32198_bbox.geojson"
OUT_DIR = Path.home() / "data/CMS/GLORYS"

MASK_CRS = "EPSG:32198"  # Québec Lambert, metres
GEOGRAPHIC_CRS = "EPSG:4326"
DENSIFY_MAX_SEGMENT_M = 1_000  # straight Lambert edges curve in lon/lat; 1 km vertices track the curve

## Mask → lon/lat bounds

Densify the edges in the projected CRS before reprojecting: reprojecting only the corners would
miss the bulge of the edges in lon/lat and understate the bounds.

In [ ]:
egsl_mask = gpd.read_file(EGSL_MASK_PATH)
assert egsl_mask.crs == MASK_CRS, egsl_mask.crs

egsl_4326 = egsl_mask.geometry.segmentize(DENSIFY_MAX_SEGMENT_M).to_crs(GEOGRAPHIC_CRS)
min_lon, min_lat, max_lon, max_lat = egsl_4326.total_bounds
print(f"lon {min_lon:.4f} → {max_lon:.4f}   lat {min_lat:.4f} → {max_lat:.4f}")

## Request spec

In [ ]:
DATASET_ID = "cmems_mod_glo_phy_my_0.083deg_P1M-m"
DATASET_VERSION = "202311"
VARIABLES = [
    "siconc", "sithick", "usi", "vsi",  # sea ice
    "uo", "vo", "thetao", "so", "zos",  # ocean surface
]
SURFACE_MAX_DEPTH_M = 0.5  # selects the top model level only (0.494 m)
START = "1993-09-01"
END = "2022-09-01"
OUT_FILENAME = "glorys12_P1M_egsl_19930901-20220901.nc"

## Login

Run once per machine: stores credentials in `~/.copernicusmarine/`, never in this notebook.

In [ ]:
copernicusmarine.login()

## Subset

`coordinates_selection_method="inside"`: keeps grid cells whose centres fall within the bounds (no buffer).
Run with `DRY_RUN = True` first to inspect the response (grid, size) before downloading.

In [ ]:
DRY_RUN = True

response = copernicusmarine.subset(
    dataset_id=DATASET_ID,
    dataset_version=DATASET_VERSION,
    variables=VARIABLES,
    minimum_longitude=min_lon,
    maximum_longitude=max_lon,
    minimum_latitude=min_lat,
    maximum_latitude=max_lat,
    minimum_depth=0,
    maximum_depth=SURFACE_MAX_DEPTH_M,
    start_datetime=START,
    end_datetime=END,
    coordinates_selection_method="inside",
    output_directory=OUT_DIR,
    output_filename=OUT_FILENAME,
    dry_run=DRY_RUN,
)
response